# Exercise 2.C: Census 2022 Real-Data Pipeline (STATA)

One applied exercise on the South Africa **Census 2022** 10% sample (STATA, about 1.3M household rows). It applies and extends the techniques from **2.2.1 to 2.2.5** on real data.

> Cleans, transforms and merges in memory, then saves one analysis table to `20_processed/`.

### Path Setup and data load (run first)

The paths and data are loaded for you. Geography loads in full; the large persons file is read in chunks, keeping only the first 50k rows. Households are loaded in Task 1, where you choose the columns.

In [ ]:
import os
import numpy as np
import pandas as pd

RAW_DATA_DIR = '../../data/0_raw/south_africa/Census2022SampleSTATA'
hh_path = ## write your code here
geo_path = os.path.join(RAW_DATA_DIR, 'Census2022Geography.dta')
persons_path = ## write your code here

# Geography loads in full
geo = pd.read_stata(geo_path)

# Persons is large, so load only the first 50k rows with a chunked reader
with pd.read_stata(persons_path, chunksize=50_000) as reader:
    persons = next(reader)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Loaded geo, persons:', geo.shape, persons.shape)

---

## Task 1: Read the descriptions, pick your columns, load

`reader.variable_labels()` returns `{column: description}`: what each of the 32 columns *is*. The names like `DERH_HSIZE` are opaque, the descriptions are not.

**Your job:**

1. Build a list `COLS` with `QID` plus the column matching each description: Household size, Sex of head of the household, Population group of head of the household, Age of head of the household, Toilet facilities, Cellphone, Access to internet, Household weight.
2. Load just those columns from `hh_path` with `pd.read_stata(hh_path, columns=COLS)` (value labels are applied by default).
3. The quantities arrived labelled (`DERH_HHAGE` as `'12'..'110'`, `DERH_HSIZE` as `'1'..'9','10 +'`), so convert `DERH_HHAGE` and `DERH_HSIZE` back to numbers (replace `'10 +'` with `'10'` first, then `pd.to_numeric`).

In [ ]:
reader = pd.io.stata.StataReader(hh_path)
var_labels = reader.variable_labels()     # {column: description}

var_labels        # read the descriptions to decide which columns to keep

In [ ]:
# Write your code here:
#  - build COLS (QID plus the 8 columns matching the descriptions above)
#  - df = pd.read_stata(hh_path, columns=COLS)
#  - convert DERH_HHAGE and DERH_HSIZE to numbers (DERH_HSIZE has a '10 +' top-code to replace first)


print('Shape:', df.shape)
df.head()

**Question:** Why pick columns with `variable_labels()`, and why did `DERH_HHAGE`/`DERH_HSIZE` still need converting?

---

## Task 2: Quick diagnostics

Get a feel for the table before transforming it.

**Your job:** show the dtypes and memory use, then a per-column summary that includes the categorical columns (transposed so it reads top to bottom).

In [ ]:
# Write your code here: show dtypes and memory usage


In [ ]:
# Write your code here: per-column summary for all columns (numeric and categorical), transposed


**Question:** Which columns are `category` and which are numeric, and why is that the split you want?

---

## Task 3: Coded missing values

The labels make missing obvious. `H12_CELLPHONE` and `H13_INTERNET_ACCESS` each have an **`"Unspecified"`** category, which means non-response. Note `H13` also has `"No access to internet services"` and `H08_TOILET` has `"None"`: those are **real** answers, not missing.

**Your job:** replace only `"Unspecified"` with `np.nan` in `H12_CELLPHONE` and `H13_INTERNET_ACCESS`, then count how many values are now missing.

In [ ]:
print(df['H12_CELLPHONE'].value_counts(dropna=False))

# Write your code here: in H12_CELLPHONE and H13_INTERNET_ACCESS, replace 'Unspecified' with np.nan


print()
print(df[['H12_CELLPHONE', 'H13_INTERNET_ACCESS']].isna().sum())

**Questions:**

- How many became `NaN`?
- Why recode `"Unspecified"` but not `"No access to internet services"` or `"None"`?

---

## Task 4: Transform with `pd.cut` and `np.select`

Derive two categorical features.

**Your job:**

- `head_age_band`: cut `DERH_HHAGE` with edges `[0, 18, 35, 50, 65, 120]` and labels `'<18', '18-34', '35-49', '50-64', '65+'`.
- `hsize_cat`: with `np.select`, label households `'Small'` (size <= 2), `'Medium'` (3 to 5), `'Large'` (> 5).

In [ ]:
# Write your code here: create df['head_age_band'] with pd.cut (5 bands) and
#                       df['hsize_cat'] with np.select ('Small'/'Medium'/'Large')


df[['QID', 'DERH_HHAGE', 'head_age_band', 'DERH_HSIZE', 'hsize_cat']].head()

**Question:** Which band does age 18 fall into?

---

## Task 5: Subset with boolean indexing

Boolean indexing `df[condition]` keeps only the matching rows. Combine conditions with `&`/`|`, each comparison in its own parentheses.

**Your job:** count households with no cellphone, and (separately) large households (size >= 6) with no cellphone.

In [ ]:
# Write your code here: select rows where H12_CELLPHONE == 'No' into `no_phone`

print('Households with no cellphone:', len(no_phone))

# Write your code here: select large households (DERH_HSIZE >= 6) with no cellphone into `large_no_phone`

print('Large households (6+) with no cellphone:', len(large_no_phone))

**Question:** What goes wrong without the inner parentheses?

---

## Task 6: Transform with `.loc` and `assign`

Two more derived columns.

**Your job:**

- `child_headed`: a boolean, `True` where the head is under 18. Start it `False`, then set the matching rows with `.loc`.
- With `df.assign(...)`, add `has_internet` (`np.nan` where `H13_INTERNET_ACCESS` is missing, otherwise `True`/`False` for "not 'No access to internet services'") and `internet_label` ('Has internet' where `has_internet == 1`, else 'No internet').

In [ ]:
df['child_headed'] = False
# Write your code here: set child_headed to True for rows where DERH_HHAGE < 18 (use .loc)

print('Child-headed households:', df['child_headed'].sum())

In [ ]:
# Write your code here: with df.assign(...), add two columns
#   has_internet   -> np.nan where H13_INTERNET_ACCESS is missing, else (H13 != 'No access to internet services')
#   internet_label -> 'Has internet' where has_internet == 1, else 'No internet'


df[['QID', 'H13_INTERNET_ACCESS', 'has_internet', 'internet_label']].head()

**Question:** Why can `internet_label` reference `x['has_internet']` created moments earlier?

---

## Task 7: Merge I, geography (one-to-one)

`geo` is already loaded (`QID, Province, District, Municipality, Geo_type`). Attach it, then add a hand-built zone lookup.

**Your job:**

1. Rename `Province` to `province_name` in `geo`, and cast `geo['QID']` to string.
2. Left-merge `geo` onto `df` on `QID`. Validate one-to-one and pass `indicator=True` to confirm every row matched.
3. Merge the provided `region_lookup` to add a `zone` column (the key has a different name on each side).

In [ ]:
# Write your code here:
#  - rename geo's Province to province_name, and cast geo['QID'] to str
#  - df = pd.merge(df, geo, on='QID', how='left', validate=..., indicator=True)


print(df['_merge'].value_counts())
df = df.drop(columns='_merge')

In [ ]:
region_lookup = pd.DataFrame({
    'prov': ['Western Cape', 'Eastern Cape', 'Northern Cape', 'KwaZulu-Natal',
             'Free State', 'North West', 'Gauteng', 'Mpumalanga', 'Limpopo'],
    'zone': ['Coastal', 'Coastal', 'Coastal', 'Coastal',
             'Inland', 'Inland', 'Inland', 'Inland', 'Inland'],
})
# Write your code here: left-merge region_lookup onto df, matching province_name (left) to prov (right),
#                       then drop the duplicate 'prov' column


df[['QID', 'province_name', 'zone']].head()

**Question:** Why is a hand-written lookup OK here, but hand-typing labels was not?

---

## Task 8: Transform strings with a lookup (`apply` with extra args)

`Series.apply(func, args=(...))` runs your function on each value and passes extra arguments too. Build `province_code(name, overrides)` that returns a short code: by default the first three letters of `name` uppercased, but a value from the `overrides` dict for the names where that rule fails (collisions or well-known abbreviations). Apply it to `province_name` to add a `province_code` column.

In [ ]:
PROVINCE_CODES = {
    'KwaZulu-Natal': 'KZN',
    'North West': 'NW',
    'Northern Cape': 'NC',
}

def province_code(name, overrides):
    # Write your code here: return overrides[name] if name is in the dict,
    #                       otherwise the first three letters of name, uppercased
    return  # Write your code here

# Write your code here: create df['province_code'] by applying province_code to df['province_name'],
#                       passing PROVINCE_CODES through args=

df[['province_name', 'province_code']].drop_duplicates().sort_values('province_name')

**Question:** Why pass `PROVINCE_CODES` through `args=` instead of hard-coding it inside the function, and what does the override dict fix that the first-three-letters rule alone gets wrong?

---

## Task 9: `apply` over rows (`axis=1`)

`df.apply(func, axis=1)` hands **each row** (a Series) to your function, which is needed when the result depends on several columns together. **Build** `deprivation_score(row)` that counts how many of three basic services a household lacks: a cellphone (`H12_CELLPHONE == 'No'`), internet (`has_internet == 0`), and a toilet (`H08_TOILET == 'None'`). Row-wise `apply` runs Python once per row, so it is slow on millions of rows: we demo it on a sample.

In [ ]:
def deprivation_score(row):
    # Count how many basic services a household lacks (0 to 3)
    # Write your code here: check the three conditions on `row` and return how many are True
    #   no cellphone  ->  row['H12_CELLPHONE'] == 'No'
    #   no internet   ->  row['has_internet'] == 0
    #   no toilet     ->  row['H08_TOILET'] == 'None'
    return  # Write your code here

sample = df.sample(5000, random_state=0)
# Write your code here: add sample['deprivation_score'] by applying the function to each row (axis=1)

sample['deprivation_score'].value_counts().sort_index()

**Question:** Why is row-wise `apply` (`axis=1`) slow on the full 1.3M rows, and what vectorised alternative could replace this function?

---

## Task 10: Merge II, aggregate persons then many-to-one

`persons` is already loaded (the first 100k rows). Aggregate it to one row per `QID`, then merge back.

**Your job:**

1. (given) Cast `persons['QID']` to string and convert `P04_AGE` to numbers.
2. Group by `QID` to build `hh_summary` with `n_persons` (count of people) and `mean_age` (their mean age).
3. Left-merge `hh_summary` onto `df` (validate one-to-one). Because only 100k person rows were loaded, `n_persons` exists for some households only, so `size_mismatch` is computed where a count exists.
4. Separately, attach each person's `province_name` from `df` with a many-to-one merge.

In [ ]:
persons['QID'] = persons['QID'].astype(str)
persons['P04_AGE'] = pd.to_numeric(persons['P04_AGE'].astype('object'))   # age came in labelled too

# Write your code here: build hh_summary grouped by QID with n_persons (size of PID) and mean_age (mean of P04_AGE)


# Write your code here: left-merge hh_summary onto df on QID (validate one-to-one)

# only the first 100k persons were loaded, so compare size where a person count exists
counted = df['n_persons'].notna()
df['size_mismatch'] = counted & (df['DERH_HSIZE'] != df['n_persons'])
print('Households matched to persons:', counted.sum())
print('Reported size != counted persons:', df['size_mismatch'].sum())
df[['QID', 'DERH_HSIZE', 'n_persons', 'mean_age']].head()

In [ ]:
# Write your code here: merge each person to their household's province_name from df[['QID','province_name']]
#                       on QID (how='left', validate many-to-one) into persons_geo

print('Persons:', len(persons), '-> after merge:', len(persons_geo))

**Question:** Why `many_to_one` here vs `one_to_one` for geography?

---

## Task 11: Post-merge validation, then append with `concat`

Check the merged table, then practise stacking rows.

**Your job:** print the number of duplicate `QID`s and the share of rows missing `province_name`. Then take the Gauteng and Western Cape subsets and stack them with `pd.concat` (reset the index).

In [ ]:
print('Rows:', len(df))
# Write your code here: print the number of duplicate QIDs, and the share of rows missing province_name


In [ ]:
# Write your code here: build the gauteng and wcape subsets (province_name equals each),
#                       then stack them with pd.concat(ignore_index=True) into `stacked`


print(len(gauteng), '+', len(wcape), '=', len(stacked))

**Question:** What should you check about inputs before stacking real waves?

---

## Task 12: Save the analysis table

Write the finished table to CSV under `20_processed/`, then read it back to confirm.

**Your job:** save `df` to `out_path` as CSV without the index, then reload it (keeping `QID` as a string).

In [ ]:
df = df.reset_index(drop=True)
PROC = '../../data/20_processed'
os.makedirs(PROC, exist_ok=True)
out_path = os.path.join(PROC, 'census2022_household_analysis_226.csv')

# Write your code here: save df to out_path as CSV without the index

print('Saved:', out_path, '|', df.shape)

In [ ]:
# Write your code here: reload out_path with pd.read_csv (keep QID as str) into `check`

print('Reloaded:', check.shape)
check.head()

**Question:** What does saving to CSV tell you about its limits?